# 02 - Train ResNet-50 Ensemble (3-slice v2)

Entraîne 5 ResNet-50 ImageNet pretrained sur le dataset 3-slice v2 (crop_margin=30). Checkpoints sauvés sous le préfixe `resnet50_wide_crop_fold_*.pth`. ~120 min sur T4 (3x plus lent que ResNet-18 v1).

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "main"
PROJECT_DIR = Path("/content/INF8225_Projet")
DRIVE_FOLDER = None
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from colab.setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())

In [ ]:
#@title Verify 3-slice v2 dataset is present
from pathlib import Path

required = [
    Path("data/classifier_dataset_resnet50_wide_crop/train/0"),
    Path("data/classifier_dataset_resnet50_wide_crop/train/1"),
    Path("data/classifier_dataset_resnet50_wide_crop/val/0"),
    Path("data/classifier_dataset_resnet50_wide_crop/val/1"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing: {missing}. Lance d'abord le notebook 01 v2."

In [ ]:
#@title Run pipeline step (ResNet-50 training)
import subprocess
import sys
subprocess.run([sys.executable, "-u", "-m", "msd_implementation.pipelines.resnet50_wide_crop.train_classifier"], check=True)

In [ ]:
#@title Inspect ResNet-50 3-slice artifacts
from msd_recall_strategy import get_resnet_checkpoint_dir

checkpoint_dir = get_resnet_checkpoint_dir()
print("checkpoint_dir =", checkpoint_dir)
for i in range(1, 6):
    ckpt = checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth"
    thr = checkpoint_dir / f"threshold_resnet50_wide_crop_run_{i}.txt"
    print(f"fold {i}: ckpt={ckpt.exists()} threshold={thr.exists()}")
assert all((checkpoint_dir / f"resnet50_wide_crop_fold_{i}.pth").exists() for i in range(1, 6)), "Missing 3-slice v2 ResNet-50 checkpoint(s)"